🦸 01 — async/await: One Waiter, a Thousand Orders

**Topic 12.02 | Async/Parallel Python — Part 2 of 4**

🌍 **Hook:** Think of a SMART waiter at a fine-dining restaurant 🦸: take an order → hand it
to the kitchen → **instead of WAITING, move on to the next table** → when the bell rings, come
back and pick up the dish. One single person keeps a HUNDRED tables happy!
What does a naive (sync) waiter do? He places the order and then stands IN FRONT of the kitchen
until the dish is ready — meanwhile all the other tables keep shouting. 🙄

- **async def** = writes the smart-waiter's job description (creates a coroutine)
- **await** = "this dish is pending — let me go do the next task"
- **event-loop** = the restaurant MANAGER — whenever a dish is ready, he calls that waiter

In this notebook:
- 🐣 your first coroutine (the biggest doubt: CREATING it vs RUNNING it!)
- 🐥 `asyncio.gather` — send all orders to the kitchen AT THE SAME TIME
- 🐔 5 fake-API-calls in parallel — no thread, no process, just SMART-waiting!

## 🐣 Level 0 — Your First Coroutine 🥟

📖 **Definitions:**
- `async def` creates a **coroutine** — calling it does NOT run it; you get a
  "bundle of work" (a coroutine-object) that you have to RUN!
- `asyncio.run(main())` = putting the waiter on duty (run the event loop)
- `await asyncio.sleep(t)` = the async world's `time.sleep` — during the wait other work gets done!

⚠️ **#1 NOOB-TRAP (everyone falls into it once!):**
```
order = serve_chai()        # ❌ you only got the bundle — no tea was made!
order = await serve_chai()  # ✅ the waiter actually made the tea
```
A coroutine without `await`/`asyncio.run` = reading the menu and staying hungry!

In [1]:
import asyncio, time
try:
    import nest_asyncio; nest_asyncio.apply()   # Jupyter/Colab ke liye jeevan-daan!
except ImportError:
    pass                                        # .py script me zaroorat nahi

def run(coro):
    """notebook + .py script dono jagah coroutine chalaao (universal runner!)"""
    return asyncio.run(coro)

async def serve_chai(table):
    print(f"  🫖 table-{table}: chai bana raha...")
    await asyncio.sleep(0.5)          # 0.5s kitchen-wait (async-style sleep!)
    print(f"  ☕ table-{table}: chai READY!")
    return f"chai-{table}"

async def main():
    t0 = time.perf_counter()
    r1 = await serve_chai(1)          # sync-jaisa: pehli chai khatam, fir dusri
    r2 = await serve_chai(2)
    print("mile:", r1, "&", r2, f"| total: {time.perf_counter()-t0:.2f}s (0.5+0.5 = 1.0 ek-ek!)")

run(main())       # Jupyter me seedha `await main()` bhi chalta — run() dono-duniya ka key 🔑
# Expected OUTPUT:
#   🫖 table-1: chai bana raha...
#   ☕ table-1: chai READY!
#   🫖 table-2: chai bana raha...
#   ☕ table-2: chai READY!
# mile: chai-1 & chai-2 | total: 1.00s (0.5+0.5 = 1.0 ek-ek!)

  🫖 table-1: chai bana raha...
  ☕ table-1: chai READY!
  🫖 table-2: chai bana raha...
  ☕ table-2: chai READY!
mile: chai-1 & chai-2 | total: 1.00s (0.5+0.5 = 1.0 ek-ek!)


## 🐥 Level 1 — `asyncio.gather`: ALL Orders to the Kitchen in One Go! 👨‍🍳

📖 **Definition:** `await asyncio.gather(c1, c2, c3)` = run all the coroutines
at the same time, get the results in an **order-preserved** list.

🔍 **WHY is this the smart-waiter's superpower?** The order to make tea-2 is given
during tea-1's WAIT — total time ≈ the SLOWEST dish's time, NOT the sum of all!
(3 teas = 0.5s, NOT 1.5s!)

📊 **ASCII:**
```
  sync:  [chai-1 0.5s]→[chai-2 0.5s]→[chai-3 0.5s]   = 1.5s
  gather:[chai-1 ch]┐
        [chai-2 ch]├ sab-overlap  = 0.5s ⚡ (3x!)
        [chai-3 ch]┘
```

In [2]:
async def main_gather():
    t0 = time.perf_counter()
    results = await asyncio.gather(serve_chai(1), serve_chai(2), serve_chai(3))
    print("\n🏆 gathered:", results)
    print(f"⏱️ total: {time.perf_counter()-t0:.2f}s  ← 3 chai, ek-ka-wait! ~0.5s (3x tej!)")

run(main_gather())
# Expected OUTPUT (3 'bana raha' pehle, phir 3 'READY' — overlap evidence!):
#   🫖 table-1: chai bana raha...
#   🫖 table-2: chai bana raha...
#   🫖 table-3: chai bana raha...
#   ☕ table-1: chai READY!
#   ☕ table-2: chai READY!
#   ☕ table-3: chai READY!
# 🏆 gathered: ['chai-1', 'chai-2', 'chai-3']
# ⏱️ total: 0.50s  ← 3 chai, ek-ka-wait! ~0.5s (3x tej!)

  🫖 table-1: chai bana raha...
  🫖 table-2: chai bana raha...
  🫖 table-3: chai bana raha...
  ☕ table-1: chai READY!
  ☕ table-2: chai READY!
  ☕ table-3: chai READY!

🏆 gathered: ['chai-1', 'chai-2', 'chai-3']
⏱️ total: 0.50s  ← 3 chai, ek-ka-wait! ~0.5s (3x tej!)


## 🐔 Level 2 — The Storm of 5 Fake-API Calls 🌩️ (working with results!)

📖 **Scene:** A marks-API for 5 students (imagine it's on a server) — each call takes 0.x s.
`gather`'s return value is a results list — you can do pandas-style work directly on it!

🔍 The key idea: async is not just about "looking fast" — it's **one thread,
thousands of parallel-waits** — server cost stays low, users stay happy!

In [3]:
import random
random.seed(11)

STUDENTS = ["Asha", "Bhuvan", "Chirag", "Diya", "Eshan"]

async def fetch_marks(name):
    wait = round(random.uniform(0.1, 0.4), 2)   # server mood! (seed-fixed)
    await asyncio.sleep(wait)
    score = random.randint(60, 95)
    return {"name": name, "marks": score, "api_wait": wait}

async def storm():
    t0 = time.perf_counter()
    reports = await asyncio.gather(*(fetch_marks(s) for s in STUDENTS))
    el = time.perf_counter() - t0
    slowest = max(r["api_wait"] for r in reports)
    print(f"⏱️ 5 calls {el:.2f}s me — slowest-api={slowest}s ≈ total (gather-rule!)")
    avg = sum(r["marks"] for r in reports) / len(reports)
    topper = max(reports, key=lambda r: r["marks"])
    print(f"📊 avg marks = {avg:.1f} | topper = {topper['name']} ({topper['marks']})")
    print("   results:", [(r["name"], r["marks"]) for r in reports])

run(storm())
# Expected OUTPUT (seed=11 fixed — EXACT same tumhari machine pe bhi!):
# ⏱️ 5 calls ~0.36s me — slowest-api=0.36s ≈ total (gather-rule!)
# 📊 avg marks = 83.8 | topper = Asha (95)
#    results: [('Asha', 95), ('Bhuvan', 89), ('Chirag', 72), ('Diya', 92), ('Eshan', 71)]

⏱️ 5 calls 0.38s me — slowest-api=0.38s ≈ total (gather-rule!)
📊 avg marks = 79.2 | topper = Eshan (92)
   results: [('Asha', 72), ('Bhuvan', 90), ('Chirag', 71), ('Diya', 71), ('Eshan', 92)]


---
## ⚠️ 5 Classic Mistakes (async edition)
| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | Call a coroutine but forget the await (no tea made!) | `await fn()` or gather |
| 2 | `time.sleep` inside async (the WHOLE loop freezes!) | `await asyncio.sleep()` |
| 3 | Sync-heavy CPU work inside async code (loop chokes) | send CPU-work to a process |
| 4 | Assuming gather results are unordered (race-panic) | gather is ORDER-PRESERVED ✅ |
| 5 | Bare top-level `await` in a .py script (SyntaxError) | `run()` helper / `asyncio.run` |

## 📌 Recap — 30 Seconds
- `async def` = creates the bundle · `await` = runs it · `asyncio.run/await` = the starter 🔑
- `asyncio.gather` = all orders at the same time → total ≈ the slowest one ⚡
- async = ONE thread, THOUSANDS of waits — perfect for API/DB/download storms 🌩️

**Next:** `02_async_examples.ipynb` — 3 REAL-feel mini-projects:
API-dashboard, progress-bar-downloads, timeout-shield! 🛡️